# 🦾 Forward and Inverse Kinematics of a 3-DOF Robotic Arm

This interactive tutorial teaches you kinematics fundamentals using the `yourdfpy` library.

**What you'll learn:**
- URDF robot description format
- Forward Kinematics (FK): Joint angles → End-effector pose
- Inverse Kinematics (IK): Target position → Joint angles
- Interactive 3D visualization with sliders

---

## Step 1: Install Dependencies

Run this cell to install the required packages:

In [1]:
!pip install yourdfpy numpy scipy plotly ipywidgets -q
print("✅ Dependencies installed!")

✅ Dependencies installed!


## Step 2: Import Libraries

In [2]:
import numpy as np
from scipy.optimize import minimize
import plotly.graph_objects as go
from ipywidgets import FloatSlider, VBox, HBox, HTML
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported!")

✅ Libraries imported!


## Step 3: Create the URDF File

A **URDF (Unified Robot Description Format)** file defines:
- **Links**: The rigid bodies of the robot
- **Joints**: The connections between links (revolute, prismatic, fixed)

Our 3-DOF arm has:
- **Joint 1**: Base rotation (around Z-axis)
- **Joint 2**: Shoulder pitch (around Y-axis)  
- **Joint 3**: Elbow pitch (around Y-axis)

In [31]:
urdf_content = '''<?xml version="1.0"?>
<robot name="arm_3dof">
  <!-- Base Link -->
  <link name="base_link">
    <visual>
      <geometry><cylinder length="0.05" radius="0.05"/></geometry>
    </visual>
  </link>

  <!-- Link 1 -->
  <link name="link1">
    <visual>
      <origin xyz="0 0 0.15" rpy="0 0 0"/>
      <geometry><cylinder length="1.12" radius="0.05"/></geometry>
    </visual>
  </link>

  <!-- Joint 1: Base to Link1 (rotate around Z) -->
  <joint name="joint1" type="revolute">
    <parent link="base_link"/>
    <child link="link1"/>
    <origin xyz="0 0 0.025" rpy="0 0 0"/>
    <axis xyz="0 0 1"/>
    <limit lower="-3.14" upper="3.14" effort="10" velocity="1"/>
  </joint>

  <!-- Link 2 -->
  <link name="link2">
    <visual>
      <origin xyz="0 0 0.125" rpy="0 0 0"/>
      <geometry><cylinder length="1.4" radius="0.05"/></geometry>
    </visual>
  </link>

  <!-- Joint 2: Link1 to Link2 (rotate around Y - shoulder) -->
  <joint name="joint2" type="revolute">
    <parent link="link1"/>
    <child link="link2"/>
    <origin xyz="0 0 0.3" rpy="0 0 0"/>
    <axis xyz="1 0 0"/>
    <limit lower="-1.57" upper="1.57" effort="10" velocity="1"/>
  </joint>

  <!-- Link 3 (End Effector) -->
  <link name="link3">
    <visual>
      <origin xyz="0 0 0.1" rpy="0 0 0"/>
      <geometry><cylinder length="0.6" radius="0.015"/></geometry>
    </visual>
  </link>

  <!-- Joint 3: Link2 to Link3 (rotate around Y - elbow) -->
  <joint name="joint3" type="revolute">
    <parent link="link2"/>
    <child link="link3"/>
    <origin xyz="0 0 0.25" rpy="0 0 0"/>
    <axis xyz="0 1 0"/>
    <limit lower="-2.0" upper="2.0" effort="10" velocity="1"/>
  </joint>

  <!-- End Effector Frame -->
  <link name="end_effector"/>
  <joint name="ee_joint" type="fixed">
    <parent link="link3"/>
    <child link="end_effector"/>
    <origin xyz="0 0 0.2" rpy="0 0 0"/>
  </joint>
</robot>
'''

# Save the URDF file
with open('arm_3dof.urdf', 'w') as f:
    f.write(urdf_content)

print("✅ URDF file created: arm_3dof.urdf")
print("\n📐 Arm Structure:")
print("   Base (0.025m) → Link1 (0.3m) → Link2 (0.25m) → Link3 (0.2m) → End Effector")
print("   Total reach: ~0.775m")

✅ URDF file created: arm_3dof.urdf

📐 Arm Structure:
   Base (0.025m) → Link1 (0.3m) → Link2 (0.25m) → Link3 (0.2m) → End Effector
   Total reach: ~0.775m


## Step 4: Load the Robot with yourdfpy

In [32]:
from yourdfpy import URDF

# Load the robot model
robot = URDF.load('arm_3dof.urdf')

print("🤖 Robot loaded successfully!")
print(f"\n📋 Robot name: {robot.robot.name}")
print(f"\n🔗 Links: {list(robot.link_map.keys())}")
print(f"\n⚙️  All joints: {list(robot.joint_map.keys())}")
print(f"\n🎮 Actuated (movable) joints: {robot.actuated_joint_names}")

# Display joint limits
print("\n📏 Joint Limits:")
for name in robot.actuated_joint_names:
    joint = robot.joint_map[name]
    print(f"   {name}: [{np.degrees(joint.limit.lower):.1f}°, {np.degrees(joint.limit.upper):.1f}°]")

🤖 Robot loaded successfully!

📋 Robot name: arm_3dof

🔗 Links: ['base_link', 'link1', 'link2', 'link3', 'end_effector']

⚙️  All joints: ['joint1', 'joint2', 'joint3', 'ee_joint']

🎮 Actuated (movable) joints: ['joint1', 'joint2', 'joint3']

📏 Joint Limits:
   joint1: [-179.9°, 179.9°]
   joint2: [-90.0°, 90.0°]
   joint3: [-114.6°, 114.6°]


## Step 5: Understand Forward Kinematics

**Forward Kinematics** computes the end-effector pose from joint angles.

Given: $\mathbf{q} = [q_1, q_2, q_3]$ (joint angles)

Compute: $\mathbf{T}_{ee} = \mathbf{T}_1(q_1) \cdot \mathbf{T}_2(q_2) \cdot \mathbf{T}_3(q_3)$ (4×4 transformation matrix)

The transformation matrix contains:
- **Position**: $\mathbf{T}_{ee}[0:3, 3]$ → $[x, y, z]$
- **Orientation**: $\mathbf{T}_{ee}[0:3, 0:3]$ → 3×3 rotation matrix

In [33]:
def forward_kinematics(robot, joint_angles, ee_link="end_effector"):
    """
    Compute end-effector pose given joint angles.
    
    Args:
        robot: URDF model
        joint_angles: list/array of joint values in radians
        ee_link: name of end-effector link
    
    Returns:
        4x4 transformation matrix (pose of end-effector)
    """
    # Convert to numpy array first
    q = np.array(joint_angles, dtype=np.float64)
    
    # Create configuration dictionary with numpy float values
    cfg = {name: np.float64(q[i]) for i, name in enumerate(robot.actuated_joint_names)}
    
    # Update robot configuration
    robot.update_cfg(cfg)
    
    # Get the transform from base to end-effector
    ee_pose = robot.get_transform(ee_link)
    
    return ee_pose


def rotation_to_euler(R):
    """Convert rotation matrix to Euler angles (roll, pitch, yaw)."""
    sy = np.sqrt(R[0, 0]**2 + R[1, 0]**2)
    singular = sy < 1e-6
    
    if not singular:
        roll = np.arctan2(R[2, 1], R[2, 2])
        pitch = np.arctan2(-R[2, 0], sy)
        yaw = np.arctan2(R[1, 0], R[0, 0])
    else:
        roll = np.arctan2(-R[1, 2], R[1, 1])
        pitch = np.arctan2(-R[2, 0], sy)
        yaw = 0
    
    return np.array([roll, pitch, yaw])


# Test forward kinematics
print("🧪 Testing Forward Kinematics\n")
print("="*50)

test_configs = [
    np.array([0, 0, 0], dtype=np.float64),           # Home position (straight up)
    np.array([0, 0.5, -0.5], dtype=np.float64),      # Slight bend
    np.array([1.57, 0.5, 0.5], dtype=np.float64),    # Rotated to side
]

for angles in test_configs:
    pose = forward_kinematics(robot, angles)
    pos = pose[:3, 3]
    euler = rotation_to_euler(pose[:3, :3])
    
    print(f"\nJoint angles (deg): [{np.degrees(angles[0]):.1f}°, {np.degrees(angles[1]):.1f}°, {np.degrees(angles[2]):.1f}°]")
    print(f"End-effector position: x={pos[0]:.3f}, y={pos[1]:.3f}, z={pos[2]:.3f}")
    print(f"Orientation (deg): roll={np.degrees(euler[0]):.1f}°, pitch={np.degrees(euler[1]):.1f}°, yaw={np.degrees(euler[2]):.1f}°")

🧪 Testing Forward Kinematics


Joint angles (deg): [0.0°, 0.0°, 0.0°]
End-effector position: x=0.000, y=0.000, z=0.775
Orientation (deg): roll=0.0°, pitch=-0.0°, yaw=0.0°

Joint angles (deg): [0.0°, 28.6°, -28.6°]
End-effector position: x=-0.096, y=-0.204, z=0.698
Orientation (deg): roll=31.9°, pitch=-24.9°, yaw=-14.7°

Joint angles (deg): [90.0°, 28.6°, 28.6°]
End-effector position: x=0.204, y=0.096, z=0.698
Orientation (deg): roll=31.9°, pitch=24.9°, yaw=104.6°


## Step 6: Implement Inverse Kinematics

**Inverse Kinematics** finds joint angles to reach a target position.

Given: Target position $\mathbf{p}_{target} = [x, y, z]$

Find: $\mathbf{q} = [q_1, q_2, q_3]$ such that $FK(\mathbf{q}) = \mathbf{p}_{target}$

We use **numerical optimization** to minimize the error:
$$\min_{\mathbf{q}} ||FK(\mathbf{q}) - \mathbf{p}_{target}||^2$$

In [34]:
def inverse_kinematics(robot, target_position, initial_guess=None, 
                       ee_link="end_effector", tol=1e-6):
    """
    Compute joint angles to reach target position using optimization.
    
    Args:
        robot: URDF model
        target_position: desired [x, y, z] position
        initial_guess: starting joint angles (optional)
        ee_link: end-effector link name
        tol: position tolerance
    
    Returns:
        joint_angles: array of joint values
        success: whether IK converged
        error: final position error
    """
    target = np.array(target_position, dtype=np.float64)
    n_joints = len(robot.actuated_joint_names)
    
    if initial_guess is None:
        initial_guess = np.zeros(n_joints, dtype=np.float64)
    else:
        initial_guess = np.array(initial_guess, dtype=np.float64)
    
    # Get joint limits
    lower_limits = []
    upper_limits = []
    for name in robot.actuated_joint_names:
        joint = robot.joint_map[name]
        lower_limits.append(joint.limit.lower)
        upper_limits.append(joint.limit.upper)
    
    bounds = list(zip(lower_limits, upper_limits))
    
    def cost_function(q):
        """Cost = squared distance to target."""
        pose = forward_kinematics(robot, q, ee_link)
        current_pos = pose[:3, 3]
        error = current_pos - target
        return np.dot(error, error)
    
    # Optimize
    result = minimize(
        cost_function,
        initial_guess,
        method='SLSQP',
        bounds=bounds,
        options={'ftol': tol, 'maxiter': 500}
    )
    
    # Verify solution
    final_pose = forward_kinematics(robot, result.x, ee_link)
    final_error = np.linalg.norm(final_pose[:3, 3] - target)
    
    return result.x, result.success and final_error < 0.01, final_error


# Test inverse kinematics
print("🧪 Testing Inverse Kinematics\n")
print("="*50)

test_targets = [
    [0.0, 0.0, 0.775],   # Straight up (home position)
    [0.2, 0.1, 0.5],     # Reach forward-right
    [0.0, 0.3, 0.4],     # Reach to side
    [-0.15, -0.15, 0.6], # Reach backward-left
]

for target in test_targets:
    solution, success, error = inverse_kinematics(robot, target)
    status = "✅" if success else "❌"
    
    # Verify
    achieved = forward_kinematics(robot, solution)[:3, 3]
    
    print(f"\n{status} Target: [{target[0]:.3f}, {target[1]:.3f}, {target[2]:.3f}]")
    print(f"   Solution (deg): [{np.degrees(solution[0]):.1f}°, {np.degrees(solution[1]):.1f}°, {np.degrees(solution[2]):.1f}°]")
    print(f"   Achieved: [{achieved[0]:.3f}, {achieved[1]:.3f}, {achieved[2]:.3f}]")
    print(f"   Error: {error*1000:.2f} mm")

🧪 Testing Inverse Kinematics


✅ Target: [0.000, 0.000, 0.775]
   Solution (deg): [0.0°, 0.0°, 0.0°]
   Achieved: [0.000, 0.000, 0.775]
   Error: 0.00 mm

✅ Target: [0.200, 0.100, 0.500]
   Solution (deg): [-2.9°, -32.3°, 102.5°]
   Achieved: [0.201, 0.100, 0.500]
   Error: 0.76 mm

❌ Target: [0.000, 0.300, 0.400]
   Solution (deg): [-0.0°, -76.0°, -0.0°]
   Achieved: [0.000, 0.437, 0.434]
   Error: 140.77 mm

✅ Target: [-0.150, -0.150, 0.600]
   Solution (deg): [22.7°, 16.3°, -79.4°]
   Achieved: [-0.150, -0.150, 0.600]
   Error: 0.37 mm


## Step 7: Compute the Jacobian (Advanced)

The **Jacobian** relates joint velocities to end-effector velocities:
$$\dot{\mathbf{x}} = \mathbf{J}(\mathbf{q}) \cdot \dot{\mathbf{q}}$$

It's useful for:
- Velocity control
- Singularity detection
- Iterative IK methods

In [35]:
def compute_jacobian(robot, joint_angles, ee_link="end_effector", delta=1e-6):
    """
    Compute the geometric Jacobian numerically (finite differences).
    
    Returns:
        J: 3xN Jacobian (linear velocity only for position IK)
    """
    n_joints = len(robot.actuated_joint_names)
    J = np.zeros((3, n_joints))
    
    q = np.array(joint_angles, dtype=np.float64)
    pose_0 = forward_kinematics(robot, q, ee_link)
    pos_0 = pose_0[:3, 3]
    
    for i in range(n_joints):
        q_plus = q.copy()
        q_plus[i] += delta
        
        pose_plus = forward_kinematics(robot, q_plus, ee_link)
        pos_plus = pose_plus[:3, 3]
        
        # Linear velocity contribution
        J[:, i] = (pos_plus - pos_0) / delta
    
    return J


def ik_jacobian(robot, target_position, initial_guess=None,
                ee_link="end_effector", max_iter=100, tol=1e-4):
    """
    Iterative IK using damped least-squares (Levenberg-Marquardt).
    """
    target = np.array(target_position, dtype=np.float64)
    n_joints = len(robot.actuated_joint_names)
    
    q = np.array(initial_guess, dtype=np.float64) if initial_guess is not None else np.zeros(n_joints, dtype=np.float64)
    damping = 0.1
    
    for iteration in range(max_iter):
        pose = forward_kinematics(robot, q, ee_link)
        current_pos = pose[:3, 3]
        
        error = target - current_pos
        error_norm = np.linalg.norm(error)
        
        if error_norm < tol:
            return q, True, error_norm, iteration
        
        # Get Jacobian
        J = compute_jacobian(robot, q, ee_link)
        
        # Damped least-squares
        JJT = J @ J.T
        dq = J.T @ np.linalg.solve(JJT + damping * np.eye(3), error)
        
        q = q + 0.5 * dq
        
        # Clamp to joint limits
        for i, name in enumerate(robot.actuated_joint_names):
            joint = robot.joint_map[name]
            q[i] = np.clip(q[i], joint.limit.lower, joint.limit.upper)
    
    final_pose = forward_kinematics(robot, q, ee_link)
    final_error = np.linalg.norm(final_pose[:3, 3] - target)
    return q, False, final_error, max_iter


# Test Jacobian-based IK
print("🧪 Testing Jacobian-based IK\n")
print("="*50)

target = [0.2, 0.15, 0.5]
solution, success, error, iters = ik_jacobian(robot, target)

print(f"Target: {target}")
print(f"Solution (deg): [{np.degrees(solution[0]):.1f}°, {np.degrees(solution[1]):.1f}°, {np.degrees(solution[2]):.1f}°]")
print(f"Converged: {success} in {iters} iterations")
print(f"Final error: {error*1000:.2f} mm")

# Show the Jacobian at this configuration
J = compute_jacobian(robot, solution)
print(f"\nJacobian at solution:\n{J.round(4)}")
print(f"\nManipulability (det(J·Jᵀ))½: {np.sqrt(np.linalg.det(J @ J.T)):.4f}")

🧪 Testing Jacobian-based IK

Target: [0.2, 0.15, 0.5]
Solution (deg): [-0.4°, -40.9°, 95.3°]
Converged: False in 100 iterations
Final error: 0.30 mm

Jacobian at solution:
[[-0.1501 -0.0013 -0.0194]
 [ 0.2003 -0.175  -0.1303]
 [ 0.      0.1516 -0.1505]]

Manipulability (det(J·Jᵀ))½: 0.0075


## Step 8: Interactive 3D Visualization 🎮

Now let's create an interactive visualization where you can:
1. **Adjust joint angles** with sliders and see the arm move
2. **Set a target position** and watch IK solve for the joint angles

In [36]:
def get_arm_points(robot, joint_angles):
    """
    Get the 3D coordinates of each joint and the end-effector.
    """
    # Convert to numpy array
    q = np.array(joint_angles, dtype=np.float64)
    
    # Create configuration dictionary with numpy float values
    cfg = {name: np.float64(q[i]) for i, name in enumerate(robot.actuated_joint_names)}
    
    robot.update_cfg(cfg)
    
    # Get positions of key points along the arm
    links = ['base_link', 'link1', 'link2', 'link3', 'end_effector']
    points = []
    
    for link in links:
        try:
            transform = robot.get_transform(link)
            points.append(transform[:3, 3])
        except:
            pass
    
    return np.array(points)


def create_3d_figure(arm_points, target=None, title="3-DOF Robotic Arm"):
    """
    Create an interactive 3D plot of the arm.
    """
    fig = go.Figure()
    
    # Arm links (thick blue line)
    fig.add_trace(go.Scatter3d(
        x=arm_points[:, 0],
        y=arm_points[:, 1],
        z=arm_points[:, 2],
        mode='lines+markers',
        line=dict(color='royalblue', width=10),
        marker=dict(size=8, color=['gray', 'blue', 'blue', 'blue', 'red']),
        name='Arm'
    ))
    
    # End effector (red sphere)
    ee_pos = arm_points[-1]
    fig.add_trace(go.Scatter3d(
        x=[ee_pos[0]],
        y=[ee_pos[1]],
        z=[ee_pos[2]],
        mode='markers',
        marker=dict(size=12, color='red', symbol='diamond'),
        name=f'End Effector ({ee_pos[0]:.3f}, {ee_pos[1]:.3f}, {ee_pos[2]:.3f})'
    ))
    
    # Target position (green sphere)
    if target is not None:
        fig.add_trace(go.Scatter3d(
            x=[target[0]],
            y=[target[1]],
            z=[target[2]],
            mode='markers',
            marker=dict(size=10, color='green', symbol='x'),
            name=f'Target ({target[0]:.3f}, {target[1]:.3f}, {target[2]:.3f})'
        ))
    
    # Base platform
    theta = np.linspace(0, 2*np.pi, 30)
    r = 0.1
    fig.add_trace(go.Scatter3d(
        x=r * np.cos(theta),
        y=r * np.sin(theta),
        z=np.zeros_like(theta),
        mode='lines',
        line=dict(color='gray', width=3),
        name='Base',
        showlegend=False
    ))
    
    # Workspace boundary (approximate sphere)
    reach = 0.75
    u = np.linspace(0, np.pi, 15)
    v = np.linspace(0, 2*np.pi, 15)
    x_sphere = reach * np.outer(np.sin(u), np.cos(v))
    y_sphere = reach * np.outer(np.sin(u), np.sin(v))
    z_sphere = reach * np.outer(np.cos(u), np.ones_like(v)) + 0.025
    
    fig.add_trace(go.Surface(
        x=x_sphere, y=y_sphere, z=z_sphere,
        opacity=0.1,
        colorscale=[[0, 'lightblue'], [1, 'lightblue']],
        showscale=False,
        name='Workspace'
    ))
    
    # Layout
    fig.update_layout(
        title=dict(text=title, x=0.5),
        scene=dict(
            xaxis=dict(range=[-0.8, 0.8], title='X (m)'),
            yaxis=dict(range=[-0.8, 0.8], title='Y (m)'),
            zaxis=dict(range=[0, 1.0], title='Z (m)'),
            aspectmode='cube',
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.0)
            )
        ),
        width=800,
        height=600,
        showlegend=True,
        legend=dict(x=0.02, y=0.98)
    )
    
    return fig


print("✅ Visualization functions defined!")

✅ Visualization functions defined!


### 8a: Interactive Forward Kinematics

Use the sliders to control joint angles and see the arm move in real-time!

In [38]:
import plotly.graph_objects as go
from ipywidgets import FloatSlider, VBox, HBox, HTML
from IPython.display import display

# Initial arm position
initial_angles = np.array([0, 0, 0], dtype=np.float64)
initial_points = get_arm_points(robot, initial_angles)

# Create FigureWidget (reactive version of Figure)
fk_fig = go.FigureWidget()

# Add arm trace
fk_fig.add_trace(go.Scatter3d(
    x=initial_points[:, 0],
    y=initial_points[:, 1],
    z=initial_points[:, 2],
    mode='lines+markers',
    line=dict(color='royalblue', width=10),
    marker=dict(size=[10, 8, 8, 8, 12], color=['gray', 'blue', 'blue', 'blue', 'red']),
    name='Arm'
))

# Add base circle
theta = np.linspace(0, 2*np.pi, 30)
fk_fig.add_trace(go.Scatter3d(
    x=0.08 * np.cos(theta),
    y=0.08 * np.sin(theta),
    z=np.zeros_like(theta),
    mode='lines',
    line=dict(color='darkgray', width=4),
    name='Base',
    showlegend=False
))

# Configure layout
fk_fig.update_layout(
    title=dict(text='Forward Kinematics | EE: (0.000, 0.000, 0.775)', x=0.5),
    scene=dict(
        xaxis=dict(range=[-0.6, 0.6], title='X (m)', backgroundcolor='white', gridcolor='lightgray'),
        yaxis=dict(range=[-0.6, 0.6], title='Y (m)', backgroundcolor='white', gridcolor='lightgray'),
        zaxis=dict(range=[0, 0.9], title='Z (m)', backgroundcolor='white', gridcolor='lightgray'),
        aspectmode='cube',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.0))
    ),
    width=750,
    height=550,
    margin=dict(l=0, r=0, t=40, b=0)
)

# Create sliders
j1_slider = FloatSlider(value=0, min=-180, max=180, step=1, description='Joint 1 (°)',
                        style={'description_width': '80px'}, layout={'width': '350px'},
                        continuous_update=True)
j2_slider = FloatSlider(value=0, min=-90, max=90, step=1, description='Joint 2 (°)',
                        style={'description_width': '80px'}, layout={'width': '350px'},
                        continuous_update=True)
j3_slider = FloatSlider(value=0, min=-115, max=115, step=1, description='Joint 3 (°)',
                        style={'description_width': '80px'}, layout={'width': '350px'},
                        continuous_update=True)

# Info display
info_html = HTML(value='<b>End Effector:</b> (0.000, 0.000, 0.775) m')

def update_fk(change):
    """Update arm visualization when sliders change."""
    j1, j2, j3 = j1_slider.value, j2_slider.value, j3_slider.value
    joint_angles = np.array([np.radians(j1), np.radians(j2), np.radians(j3)], dtype=np.float64)
    arm_points = get_arm_points(robot, joint_angles)
    ee = arm_points[-1]
    
    # Update the arm trace data (using batch_update for performance)
    with fk_fig.batch_update():
        fk_fig.data[0].x = arm_points[:, 0]
        fk_fig.data[0].y = arm_points[:, 1]
        fk_fig.data[0].z = arm_points[:, 2]
        fk_fig.layout.title.text = f'Forward Kinematics | EE: ({ee[0]:.3f}, {ee[1]:.3f}, {ee[2]:.3f})'
    
    info_html.value = f'<b>End Effector:</b> ({ee[0]:.3f}, {ee[1]:.3f}, {ee[2]:.3f}) m &nbsp;&nbsp; <b>Joints:</b> [{j1:.0f}°, {j2:.0f}°, {j3:.0f}°]'

# Connect sliders to update function
j1_slider.observe(update_fk, names='value')
j2_slider.observe(update_fk, names='value')
j3_slider.observe(update_fk, names='value')

# Display widgets
slider_box = VBox([j1_slider, j2_slider, j3_slider])
display(VBox([slider_box, info_html, fk_fig]))

print("🎮 Drag the sliders to move the arm!")

🎮 Drag the sliders to move the arm!


### 8b: Interactive Inverse Kinematics

Set a target position and watch the IK solver find the joint angles!

In [40]:
# Initial target and IK solution
init_target = np.array([0.2, 0.1, 0.5], dtype=np.float64)
init_solution, init_success, init_error = inverse_kinematics(robot, init_target)
init_arm_points = get_arm_points(robot, init_solution)

# Create FigureWidget for IK
ik_fig = go.FigureWidget()

# Add arm trace
ik_fig.add_trace(go.Scatter3d(
    x=init_arm_points[:, 0],
    y=init_arm_points[:, 1],
    z=init_arm_points[:, 2],
    mode='lines+markers',
    line=dict(color='royalblue', width=10),
    marker=dict(size=[10, 8, 8, 8, 12], color=['gray', 'blue', 'blue', 'blue', 'red']),
    name='Arm'
))

# Add target marker
ik_fig.add_trace(go.Scatter3d(
    x=[init_target[0]],
    y=[init_target[1]],
    z=[init_target[2]],
    mode='markers',
    marker=dict(size=12, color='green', symbol='x'),
    name='Target'
))

# Add base circle
theta = np.linspace(0, 2*np.pi, 30)
ik_fig.add_trace(go.Scatter3d(
    x=0.08 * np.cos(theta),
    y=0.08 * np.sin(theta),
    z=np.zeros_like(theta),
    mode='lines',
    line=dict(color='darkgray', width=4),
    showlegend=False
))

# Configure layout
ik_fig.update_layout(
    title=dict(text='Inverse Kinematics ✅', x=0.5),
    scene=dict(
        xaxis=dict(range=[-0.6, 0.6], title='X (m)', backgroundcolor='white', gridcolor='lightgray'),
        yaxis=dict(range=[-0.6, 0.6], title='Y (m)', backgroundcolor='white', gridcolor='lightgray'),
        zaxis=dict(range=[0, 0.9], title='Z (m)', backgroundcolor='white', gridcolor='lightgray'),
        aspectmode='cube',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.0))
    ),
    width=750,
    height=550,
    margin=dict(l=0, r=0, t=40, b=0)
)

# Create sliders for target position
x_slider = FloatSlider(value=0.2, min=-0.5, max=0.5, step=0.02, description='Target X (m)',
                       style={'description_width': '90px'}, layout={'width': '350px'},
                       continuous_update=True)
y_slider = FloatSlider(value=0.1, min=-0.5, max=0.5, step=0.02, description='Target Y (m)',
                       style={'description_width': '90px'}, layout={'width': '350px'},
                       continuous_update=True)
z_slider = FloatSlider(value=0.5, min=0.1, max=0.8, step=0.02, description='Target Z (m)',
                       style={'description_width': '90px'}, layout={'width': '350px'},
                       continuous_update=True)

# Info display
ik_info = HTML(value='<b>Status:</b> ✅ Solved | <b>Error:</b> 0.00 mm')

# Store last solution for warm-starting IK
last_solution = [init_solution.copy()]

def update_ik(change):
    """Update IK visualization when target sliders change."""
    target = np.array([x_slider.value, y_slider.value, z_slider.value], dtype=np.float64)
    
    # Solve IK (warm-start from last solution)
    solution, success, error = inverse_kinematics(robot, target, initial_guess=last_solution[0])
    last_solution[0] = solution.copy()
    
    arm_points = get_arm_points(robot, solution)
    
    status = '✅ Solved' if success else '❌ Failed'
    angles = f'[{np.degrees(solution[0]):.1f}°, {np.degrees(solution[1]):.1f}°, {np.degrees(solution[2]):.1f}°]'
    
    # Update figure
    with ik_fig.batch_update():
        # Update arm
        ik_fig.data[0].x = arm_points[:, 0]
        ik_fig.data[0].y = arm_points[:, 1]
        ik_fig.data[0].z = arm_points[:, 2]
        # Update target
        ik_fig.data[1].x = [target[0]]
        ik_fig.data[1].y = [target[1]]
        ik_fig.data[1].z = [target[2]]
        # Update title
        ik_fig.layout.title.text = f'Inverse Kinematics {"✅" if success else "❌"}'
    
    ik_info.value = f'<b>Status:</b> {status} | <b>Error:</b> {error*1000:.2f} mm | <b>Joints:</b> {angles}'

# Connect sliders
x_slider.observe(update_ik, names='value')
y_slider.observe(update_ik, names='value')
z_slider.observe(update_ik, names='value')

# Display
ik_slider_box = VBox([x_slider, y_slider, z_slider])
display(VBox([ik_slider_box, ik_info, ik_fig]))

print("🎯 Move the target sliders and watch the arm reach for it!")

🎯 Move the target sliders and watch the arm reach for it!


## Step 9: Workspace Visualization

Let's visualize the **reachable workspace** by sampling many configurations.

In [42]:
def compute_workspace(robot, n_samples=2000):
    """
    Sample the reachable workspace of the arm.
    """
    # Get joint limits
    limits = []
    for name in robot.actuated_joint_names:
        joint = robot.joint_map[name]
        limits.append((joint.limit.lower, joint.limit.upper))
    
    # Random sampling
    positions = []
    for _ in range(n_samples):
        # Random joint angles within limits (as numpy float64)
        q = np.array([np.random.uniform(l[0], l[1]) for l in limits], dtype=np.float64)
        pose = forward_kinematics(robot, q)
        positions.append(pose[:3, 3])
    
    return np.array(positions)


print("Computing workspace (sampling 3000 configurations)...")
workspace = compute_workspace(robot, n_samples=3000)

# Create workspace visualization
fig = go.Figure()

# Workspace points (color by height)
fig.add_trace(go.Scatter3d(
    x=workspace[:, 0],
    y=workspace[:, 1],
    z=workspace[:, 2],
    mode='markers',
    marker=dict(
        size=2,
        color=workspace[:, 2],
        colorscale='Viridis',
        colorbar=dict(title='Z height (m)')
    ),
    name='Reachable Points'
))

# Add arm at home position
home_points = get_arm_points(robot, np.array([0, 0, 0], dtype=np.float64))
fig.add_trace(go.Scatter3d(
    x=home_points[:, 0],
    y=home_points[:, 1],
    z=home_points[:, 2],
    mode='lines+markers',
    line=dict(color='red', width=8),
    marker=dict(size=6, color='red'),
    name='Arm (home position)'
))

fig.update_layout(
    title='3-DOF Arm Reachable Workspace',
    scene=dict(
        xaxis=dict(title='X (m)'),
        yaxis=dict(title='Y (m)'),
        zaxis=dict(title='Z (m)'),
        aspectmode='cube'
    ),
    width=800,
    height=600
)

fig.show()

print(f"\n📊 Workspace Statistics:")
print(f"   X range: [{workspace[:, 0].min():.3f}, {workspace[:, 0].max():.3f}] m")
print(f"   Y range: [{workspace[:, 1].min():.3f}, {workspace[:, 1].max():.3f}] m")
print(f"   Z range: [{workspace[:, 2].min():.3f}, {workspace[:, 2].max():.3f}] m")

Computing workspace (sampling 3000 configurations)...



📊 Workspace Statistics:
   X range: [-0.439, 0.449] m
   Y range: [-0.443, 0.443] m
   Z range: [0.325, 0.775] m


## Step 10: Trajectory Planning

Plan a smooth trajectory from one point to another.

In [43]:
def plan_trajectory(robot, start_pos, end_pos, n_points=20):
    """
    Plan a linear trajectory in Cartesian space.
    """
    # Interpolate positions
    t = np.linspace(0, 1, n_points)
    positions = np.array([start_pos + (end_pos - start_pos) * ti for ti in t])
    
    # Solve IK for each point
    trajectory = []
    prev_solution = None
    
    for i, pos in enumerate(positions):
        solution, success, error = inverse_kinematics(
            robot, pos, 
            initial_guess=prev_solution
        )
        trajectory.append({
            'position': pos,
            'joints': solution,
            'success': success,
            'error': error
        })
        prev_solution = solution
    
    return trajectory


# Plan a trajectory
start = np.array([0.3, 0.0, 0.5])
end = np.array([-0.2, 0.2, 0.6])

print(f"Planning trajectory from {start} to {end}...")
trajectory = plan_trajectory(robot, start, end, n_points=15)

# Visualize trajectory
fig = go.Figure()

# Draw arm at each waypoint
colors = ['rgba(0,0,255,' + str(0.3 + 0.7*i/len(trajectory)) + ')' for i in range(len(trajectory))]

for i, wp in enumerate(trajectory):
    arm_points = get_arm_points(robot, wp['joints'])
    fig.add_trace(go.Scatter3d(
        x=arm_points[:, 0],
        y=arm_points[:, 1],
        z=arm_points[:, 2],
        mode='lines',
        line=dict(color=colors[i], width=4),
        showlegend=False
    ))

# End effector path
ee_path = np.array([wp['position'] for wp in trajectory])
fig.add_trace(go.Scatter3d(
    x=ee_path[:, 0],
    y=ee_path[:, 1],
    z=ee_path[:, 2],
    mode='lines+markers',
    line=dict(color='red', width=4),
    marker=dict(size=4, color='red'),
    name='End Effector Path'
))

# Start and end markers
fig.add_trace(go.Scatter3d(
    x=[start[0]], y=[start[1]], z=[start[2]],
    mode='markers',
    marker=dict(size=12, color='green', symbol='circle'),
    name='Start'
))
fig.add_trace(go.Scatter3d(
    x=[end[0]], y=[end[1]], z=[end[2]],
    mode='markers',
    marker=dict(size=12, color='orange', symbol='diamond'),
    name='End'
))

fig.update_layout(
    title='Trajectory Planning: Linear Path in Cartesian Space',
    scene=dict(
        xaxis=dict(range=[-0.6, 0.6], title='X (m)'),
        yaxis=dict(range=[-0.6, 0.6], title='Y (m)'),
        zaxis=dict(range=[0, 1.0], title='Z (m)'),
        aspectmode='cube'
    ),
    width=800,
    height=600
)

fig.show()

# Show joint trajectories
print("\n📈 Joint Trajectories:")
joint_traj = np.array([wp['joints'] for wp in trajectory])

fig2 = go.Figure()
for i, name in enumerate(robot.actuated_joint_names):
    fig2.add_trace(go.Scatter(
        y=np.degrees(joint_traj[:, i]),
        mode='lines+markers',
        name=name
    ))

fig2.update_layout(
    title='Joint Angles Along Trajectory',
    xaxis_title='Waypoint',
    yaxis_title='Angle (degrees)',
    width=700,
    height=400
)
fig2.show()

Planning trajectory from [0.3 0.  0.5] to [-0.2  0.2  0.6]...



📈 Joint Trajectories:


## Summary

Congratulations! You've learned:

| Topic | What You Learned |
|-------|------------------|
| **URDF** | Robot description format with links and joints |
| **yourdfpy** | Loading and manipulating robot models |
| **Forward Kinematics** | Computing end-effector pose from joint angles |
| **Inverse Kinematics** | Finding joint angles to reach a target (optimization & Jacobian methods) |
| **Jacobian** | Relating joint and Cartesian velocities |
| **Workspace** | Visualizing reachable positions |
| **Trajectory Planning** | Moving smoothly between positions |

### Next Steps

1. **Add more DOF**: Extend the arm to 6-DOF for full pose control
2. **Collision avoidance**: Add obstacle detection
3. **Dynamics**: Add inertia and compute torques
4. **Real robot**: Connect to hardware via ROS

---

## Bonus: Animation

Create an animated visualization of the trajectory!

In [44]:
def create_animation(robot, trajectory):
    """
    Create an animated plot of the trajectory.
    """
    # Get all arm positions
    frames = []
    for i, wp in enumerate(trajectory):
        arm_points = get_arm_points(robot, wp['joints'])
        frames.append(go.Frame(
            data=[
                go.Scatter3d(
                    x=arm_points[:, 0],
                    y=arm_points[:, 1],
                    z=arm_points[:, 2],
                    mode='lines+markers',
                    line=dict(color='royalblue', width=10),
                    marker=dict(size=8, color=['gray', 'blue', 'blue', 'blue', 'red'])
                )
            ],
            name=str(i)
        ))
    
    # Initial arm position
    initial_points = get_arm_points(robot, trajectory[0]['joints'])
    
    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=initial_points[:, 0],
                y=initial_points[:, 1],
                z=initial_points[:, 2],
                mode='lines+markers',
                line=dict(color='royalblue', width=10),
                marker=dict(size=8, color=['gray', 'blue', 'blue', 'blue', 'red']),
                name='Arm'
            ),
            # End effector path (static)
            go.Scatter3d(
                x=[wp['position'][0] for wp in trajectory],
                y=[wp['position'][1] for wp in trajectory],
                z=[wp['position'][2] for wp in trajectory],
                mode='lines',
                line=dict(color='rgba(255,0,0,0.5)', width=2, dash='dash'),
                name='Path'
            )
        ],
        frames=frames
    )
    
    # Add play/pause buttons
    fig.update_layout(
        title='🎬 Animated Trajectory',
        scene=dict(
            xaxis=dict(range=[-0.6, 0.6], title='X (m)'),
            yaxis=dict(range=[-0.6, 0.6], title='Y (m)'),
            zaxis=dict(range=[0, 1.0], title='Z (m)'),
            aspectmode='cube'
        ),
        updatemenus=[
            dict(
                type='buttons',
                showactive=False,
                y=0,
                x=0.1,
                xanchor='right',
                yanchor='top',
                buttons=[
                    dict(
                        label='▶ Play',
                        method='animate',
                        args=[None, {
                            'frame': {'duration': 200, 'redraw': True},
                            'fromcurrent': True,
                            'transition': {'duration': 100}
                        }]
                    ),
                    dict(
                        label='⏸ Pause',
                        method='animate',
                        args=[[None], {
                            'frame': {'duration': 0, 'redraw': False},
                            'mode': 'immediate',
                            'transition': {'duration': 0}
                        }]
                    )
                ]
            )
        ],
        sliders=[{
            'active': 0,
            'steps': [
                {'args': [[str(i)], {'frame': {'duration': 0, 'redraw': True}, 'mode': 'immediate'}],
                 'label': str(i), 'method': 'animate'}
                for i in range(len(trajectory))
            ],
            'x': 0.1,
            'len': 0.8,
            'y': -0.05,
            'currentvalue': {'prefix': 'Waypoint: ', 'visible': True, 'xanchor': 'center'},
            'transition': {'duration': 100}
        }],
        width=800,
        height=650
    )
    
    return fig


# Create and show animation
anim_fig = create_animation(robot, trajectory)
anim_fig.show()

print("\n🎮 Use the Play/Pause buttons or drag the slider to control the animation!")


🎮 Use the Play/Pause buttons or drag the slider to control the animation!
